In [0]:
%run ./config

In [0]:
r_df=spark.read.parquet(silver+'/races_tb')
c_df=spark.read.parquet(silver+'/circuits_tb')

In [0]:
r_c_df=r_df.join(c_df,r_df.circuit_id==c_df.circuit_id,'inner')

r_c_df=r_c_df.select(r_df.race_id,  
                     r_df.circuit_id, 
                     r_df.name.alias('race_name'), 
                     r_df.race_timestamp, 
                     r_df.race_year,
                     c_df.circuit_ref, 
                     c_df.name.alias('circuit_name'), 
                     c_df.location, 
                     c_df.country)

In [0]:
d_df=spark.read.parquet(silver+'/drivers_tb')
cn_df=spark.read.parquet(silver+'/constructors')
rs_df=spark.read.parquet(silver+'/results')

In [0]:
j_df= cn_df.join(rs_df, rs_df.constructor_id==cn_df.constructor_id,'inner')\
            .join(d_df, d_df.driver_id==rs_df.driver_id, 'inner')\
            .select(
                    d_df.name.alias('driver_name'),
                    d_df.nationality.alias('driver_nationality'),
                    d_df.number.alias('driver_number'),
                    d_df.code.alias('driver_code'),
                    d_df.driver_ref,
                    d_df.driver_id,
                    cn_df.constructor_id, 
                    cn_df.constructor_ref, 
                    cn_df.name, 
                    cn_df.nationality, 
                    rs_df.result_id, 
                    rs_df.race_id,
                    rs_df.number, 
                    rs_df.grid, 
                    rs_df.position, 
                    rs_df.position_order, 
                    rs_df.points, 
                    rs_df.laps, 
                    rs_df.time, 
                    rs_df.milliseconds, 
                    rs_df.fastest_lap, 
                    rs_df.rank, 
                    rs_df.fastest_lap_time, 
                    rs_df.fastest_lap_speed, 
                    rs_df.ingestion_date)

In [0]:
df=r_c_df.join(j_df, r_c_df.race_id==j_df.race_id,'inner')
f_df=df.select(
             r_c_df.race_id,  
             r_c_df.circuit_id, 
             r_c_df.race_name, 
             r_c_df.race_timestamp, 
             r_c_df.race_year,
             r_c_df.circuit_ref, 
             r_c_df.circuit_name, 
             r_c_df.location, 
             r_c_df.country,
             j_df.constructor_id, 
             j_df.constructor_ref, 
             j_df.name.alias('constructor_name'), 
             j_df.nationality, 
             j_df.result_id, 
             j_df.driver_id, 
             j_df.number, 
             j_df.grid, 
             j_df.position, 
             j_df.position_order, 
             j_df.points, 
             j_df.laps, 
             j_df.time, 
             j_df.milliseconds, 
             j_df.fastest_lap, 
             j_df.rank, 
             j_df.fastest_lap_time, 
             j_df.fastest_lap_speed,
             j_df.driver_name,
             j_df.driver_nationality,
             j_df.driver_number,
             j_df.driver_code,
             j_df.driver_ref)

In [0]:
result_df = f_df.select(
    f_df.race_year,
    f_df.race_name,
    f_df.location.alias('race_location'),
    f_df.driver_name, 
    f_df.driver_number,
    f_df.driver_nationality,
    f_df.constructor_name.alias('team'),
    f_df.position, 
    f_df.grid,
    f_df.fastest_lap,
    f_df.points,
    f_df.race_timestamp
).withColumn('race_date', to_date(col('race_timestamp')))\
 .withColumn('race_time', date_format(col('race_timestamp'), 'HH:mm:ss'))\
 .withColumn('created_date', current_timestamp())\
 .drop('race_timestamp')


In [0]:
# result_df.filter("race_date='2020-12-13'").orderBy(col('points').desc())
result_df.write.mode('overwrite').partitionBy('race_year').parquet(gold+'/race_results')